# Day 16 · 老前輩的實力：Template Workflow Agents

> 第三部・戰術編排　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 16 - 老前輩的實力：Template Workflow Agents.md`

> 📌 **這一天建議放在 [Day 13](../day13_graph_workflows/) 之前讀。**
> Sequential / Parallel / Loop 是跨語言（Python / Go / TypeScript）共通的原語，
> 概念比 Graph 單純，是理解圖的墊腳石。

## 今天要學會

1. 用 `SequentialAgent` / `ParallelAgent` / `LoopAgent` 組流程
2. 知道 **`LoopAgent` 的停止責任在誰身上**
3. 三個都不夠用時怎麼繼承 `BaseAgent`

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 0. ⚠️ 先講版本狀態

這三個 agent 在 **ADK Python 2.8.0 已被標記為 deprecated**。

那為什麼還要學？三個理由：

1. **它們仍然可用**，而且大量既有專案在用
2. **它們是跨語言共通的原語**——Go 和 TypeScript 沒有 Graph Workflow
3. **概念比圖單純**，先懂這三個再看 Day 13 會輕鬆很多

本日每個範例都會標註對應的 Graph 寫法。

In [2]:
import google.adk

print("ADK 版本:", google.adk.__version__)

from google.adk.agents import LoopAgent, ParallelAgent, SequentialAgent

for cls in (SequentialAgent, ParallelAgent, LoopAgent):
    print(f"  {cls.__name__:16s} 仍可 import ✅")

ADK 版本: 2.8.0
  SequentialAgent  仍可 import ✅
  ParallelAgent    仍可 import ✅
  LoopAgent        仍可 import ✅


## 1. `SequentialAgent`：靠 Output Key 傳資料

最單純的模式：sub_agents 依序執行。

**資料怎麼流**：前一個用 `output_key` 寫進 state，後一個用 `{key?}` 讀出來。
這跟 Graph 的節點參數綁定是**兩套不同機制**（Day 14）。

In [3]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner

classifier = LlmAgent(
    name="classifier",
    model=get_model(),
    instruction=(
        "你是客訴分類員。判斷抱怨屬於「物流」「品質」「客服態度」「價格」哪一類，"
        "只回類別，不要其他內容。"
    ),
    output_key="category",      # ← 寫進 state["category"]
)

drafter = LlmAgent(
    name="drafter",
    model=get_model(),
    instruction=(
        "你是客服。這是一則「{category?}」類的客訴。\n"   # ← 從 state 讀
        "針對抱怨寫一段道歉與處理方式，三句話以內，繁體中文。"
    ),
    output_key="draft",
)

polisher = LlmAgent(
    name="polisher",
    model=get_model(),
    instruction="把這段客服回覆潤飾得更誠懇專業，保持三句話以內，只回內容：\n\n{draft?}",
    output_key="final_reply",
)

pipeline = SequentialAgent(name="pipeline", sub_agents=[classifier, drafter, polisher])
print("順序:", [a.name for a in pipeline.sub_agents])

順序: ['classifier', 'drafter', 'polisher']


In [4]:
runner = InMemoryRunner(agent=pipeline, app_name="day16")
sid = await new_session(runner)

complaint = "我上週訂的東西到現在還沒收到，查物流也沒更新，打客服電話都沒人接。"
await ask(runner, complaint, session_id=sid, trace=True)

print("\n--- 每一站留在 state 的東西 ---")
print_state(await peek_state(runner, sid))

  💬 [classifier] 物流


  💬 [drafter] 對於物流延誤與客服電話未接的情況，我們深感抱歉。我們已緊急聯繫貨運公司查明包裹下落，將於今日內回報您最新進度並優先安排配送。


  💬 [polisher] 對於物流延誤與電話未能即時接聽，我們致上最深的歉意。我們已全面追查包裹下落，將於今日內親自向您回報進度，並為您安排優先配送。造成您的困擾，我們銘感抱歉，懇請您再給我們一次服務的機會。

--- 每一站留在 state 的東西 ---
  category: 物流
  draft: 對於物流延誤與客服電話未接的情況，我們深感抱歉。我們已緊急聯繫貨運公司查明包裹下落，將於今日內回報您最新進度並優先安排配送。
  final_reply: 對於物流延誤與電話未能即時接聽，我們致上最深的歉意。我們已全面追查包裹下落，將於今日內親自向您回報進度，並為您安排優先配送。造成您的困擾，我們銘感抱歉，懇請您再給我們一次服務的機會。


### ⚠️ `{key?}` 的問號很重要

少了問號，state 剛好沒那個 key 時會整個炸掉。
而在 pipeline 裡，**任何一站失敗都會讓後面全部拿不到資料**。

In [5]:
strict = LlmAgent(
    name="strict",
    model=get_model(),
    instruction="根據分類 {category} 回覆。",   # ← 沒有問號
)
strict_runner = InMemoryRunner(agent=strict, app_name="day16")
sid_s = await new_session(strict_runner)       # 空 state
try:
    await ask(strict_runner, "你好", session_id=sid_s)
    print("沒有報錯")
except Exception as exc:
    print(f"❌ {type(exc).__name__}: {str(exc)[:120]}")

❌ KeyError: 'Context variable not found: `category`.'


> **Graph 對應寫法**：`Workflow(edges=[(START, a), (a, b), (b, c)])`，
> 資料靠節點參數從 state 綁定（Day 13 / 14）。

## 2. `ParallelAgent`：同時跑，但分支互不相通

**關鍵限制**：平行的分支**看不到彼此的 state**。它們同時起跑，誰也讀不到誰。

In [6]:
def make_reviewers() -> list[LlmAgent]:
    """每次呼叫都給一組全新實例。

    ⚠️ ADK 規定「一個 agent 只能有一個父節點」，
    所以同一批 agent 不能同時掛在 ParallelAgent 和 SequentialAgent 底下。
    """
    return [
        LlmAgent(name="pros", model=get_model(),
                 instruction="列出這個方案的三個優點，條列式，繁體中文。",
                 output_key="pros"),
        LlmAgent(name="cons", model=get_model(),
                 instruction="列出這個方案的三個風險，條列式，繁體中文。",
                 output_key="cons"),
        LlmAgent(name="cost", model=get_model(),
                 instruction="估算這個方案的主要成本項目，三點以內，繁體中文。",
                 output_key="cost"),
    ]


board = ParallelAgent(name="board", sub_agents=make_reviewers())

TOPIC = "我們公司要不要把所有內部工具都改成 AI Agent 驅動？"

In [7]:
import time

r_par = InMemoryRunner(agent=board, app_name="day16")
sid_p = await new_session(r_par)
t0 = time.perf_counter()
await ask(r_par, TOPIC, session_id=sid_p)
par_secs = time.perf_counter() - t0

print(f"平行耗時 {par_secs:.1f} 秒\n")
print_state(await peek_state(r_par, sid_p), max_len=70)

平行耗時 19.6 秒

  pros: 將公司所有內部工具改為 AI Agent 驅動，具有以下三個主要優點：  1. **大幅提升工作效率與自動化程度**    AI Agent …（共 382 字）
  cost: 評估將所有內部工具改為 AI Agent 驅動的方案，主要成本項目如下：  1. **API 呼叫與基礎設施運算成本**：大量使用 LLM  …（共 312 字）
  cons: 我是內部風險評估代理人 (cons)。針對「將所有內部工具改為 AI Agent 驅動」的方案，以下列出三個主要風險：  *   **資安與 …（共 412 字）


### ⚠️ 一個 agent 只能有一個父節點

想把同一批 agent 換個編排方式跑，**不能重用實例**：

In [8]:
try:
    SequentialAgent(name="reuse", sub_agents=board.sub_agents)
except Exception as exc:
    print(f"❌ {type(exc).__name__}")
    print(f"   {str(exc)[:170]}")

print("\n→ 所以實務上 agent 常寫成工廠函式，而不是模組層級的變數。")

❌ ValidationError
   1 validation error for SequentialAgent
  Value error, Agent `pros` already has a parent agent, current parent: `board`, trying to add: `reuse` [type=value_error, input_va

→ 所以實務上 agent 常寫成工廠函式，而不是模組層級的變數。


In [9]:
serial = SequentialAgent(name="serial_board", sub_agents=make_reviewers())   # 新的一組
r_seq = InMemoryRunner(agent=serial, app_name="day16")
sid_q = await new_session(r_seq)
t0 = time.perf_counter()
await ask(r_seq, TOPIC, session_id=sid_q)
seq_secs = time.perf_counter() - t0

print(f"平行: {par_secs:.1f} 秒")
print(f"依序: {seq_secs:.1f} 秒")
print("\n（本教材有 8 RPM 的全域節流，會壓縮平行的優勢；實際差距更大。）")

平行: 19.6 秒
依序: 25.2 秒

（本教材有 8 RPM 的全域節流，會壓縮平行的優勢；實際差距更大。）


### 平行完之後要有人收尾

`ParallelAgent` 只負責「同時跑完」，它不會幫你整合。
標準骨架是外面再包一層 `SequentialAgent`：**先平行、再匯總**。

In [10]:
synthesizer = LlmAgent(
    name="synthesizer",
    model=get_model(),
    instruction=(
        "你是決策顧問。根據以下三份分析給出明確建議（做／不做／有條件做）"
        "並說明理由，五句話以內，繁體中文。\n\n"
        "【優點】\n{pros?}\n\n【風險】\n{cons?}\n\n【成本】\n{cost?}"
    ),
    output_key="decision",
)

full_board = SequentialAgent(
    name="full_board",
    sub_agents=[
        ParallelAgent(name="board2", sub_agents=make_reviewers()),   # 又一組新的
        synthesizer,
    ],
)

r_full = InMemoryRunner(agent=full_board, app_name="day16")
sid_f = await new_session(r_full)
await ask(r_full, TOPIC, session_id=sid_f)
print("=== 最終建議 ===")
print((await peek_state(r_full, sid_f)).get("decision"))

=== 最終建議 ===
建議：**有條件做**。

理由：雖然 AI Agent 能顯著提升跨工具協作效率與自動化工作流程，但全面導入伴隨著高昂的維護成本、資安隱私風險以及幻覺帶來的業務中斷危機。因此，建議採取漸進式策略，優先在低風險的非核心環節試點，並嚴格控管 AI 的自主執行權限，確保成本與風險在可控範圍內。


這是最常用的多 agent 骨架：**fan-out（平行展開）→ join（匯總）**。

> **Graph 對應寫法**：`(START, (a, b, c))` fan-out + `JoinNode` join（Day 13）。
> Graph 的優勢是可以**細緻控制**誰等誰；`ParallelAgent` 只能整批平行。

## 3. `LoopAgent`：停止不是它的責任

這是本日最重要的一句話：

> **`LoopAgent` 自己不知道什麼時候該停。**
> 停止是 sub_agent 的責任——某個 agent 要呼叫 `exit_loop` 工具。

In [11]:
from google.adk.tools import exit_loop

generator = LlmAgent(
    name="slogan_generator",
    model=get_model(),
    instruction=(
        "你是文案。為「給工程師的保溫杯」寫一句廣告標語。\n"
        "如果下面有前一版和評語，請根據評語改寫；沒有就寫第一版。只回標語本身。\n\n"
        "前一版：{slogan?}\n評語：{critique?}"
    ),
    output_key="slogan",
)

critic = LlmAgent(
    name="slogan_critic",
    model=get_model(),
    instruction=(
        "你是嚴格的創意總監。評價這句標語：{slogan?}\n\n"
        "如果它同時做到「16 字以內」「有具體畫面」「不用『科技』『創新』這類空話」，"
        "就**呼叫 exit_loop 工具**結束流程。\n"
        "否則用一句話指出最該改的地方，不要呼叫工具。"
    ),
    tools=[exit_loop],
    output_key="critique",
)

refine = LoopAgent(
    name="refine",
    sub_agents=[generator, critic],
    max_iterations=4,     # 安全閥
)

r_loop = InMemoryRunner(agent=refine, app_name="day16")
sid_l = await new_session(r_loop)
await ask(r_loop, "開始", session_id=sid_l, trace=True)

print(f"\n=== 最終標語 ===\n{(await peek_state(r_loop, sid_l)).get('slogan')}")

  💬 [slogan_generator] 程式會當機，咖啡不能涼。


  🔧 [slogan_critic] 呼叫 exit_loop({})
  ↩️  [slogan_critic] exit_loop 回傳 {'result': None}

=== 最終標語 ===
程式會當機，咖啡不能涼。


### `max_iterations` 是你唯一的安全閥

拿掉 `exit_loop`、或 critic 永遠不滿意，`LoopAgent` 就會一直跑。

**一輪 = 2 個 agent = 至少 2 次模型呼叫。** `max_iterations=10` 就是 20 次。

In [12]:
runaway = LoopAgent(
    name="runaway",
    sub_agents=[
        LlmAgent(name="counter", model=get_model(),
                 instruction="說一個數字，只回數字。前一個是 {n?}",
                 output_key="n"),
    ],
    max_iterations=3,     # ← 沒有 exit_loop，全靠這行
)

r_run = InMemoryRunner(agent=runaway, app_name="day16")
sid_r = await new_session(r_run)
await ask(r_run, "開始", session_id=sid_r, trace=True)
print("\n跑滿 3 輪被 max_iterations 擋下來——不是它自己想停的。")

  💬 [counter] 1


  💬 [counter] 2


  💬 [counter] 3

跑滿 3 輪被 max_iterations 擋下來——不是它自己想停的。


> **Graph 對應寫法**：邊可以指回前面的節點形成迴圈，
> 停止條件用 `ctx.route` 判斷（確定性），不需要模型呼叫工具。
> 這是 Graph 比 `LoopAgent` 可靠的地方。

## 4. 三個都不夠用時：繼承 `BaseAgent`

需要「條件分支」「動態決定跑幾次」時，這三個都做不到。
舊做法是自己繼承 `BaseAgent` 實作 `_run_async_impl`。

In [13]:
from typing import AsyncGenerator

from google.adk.agents import BaseAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event


class ConditionalAgent(BaseAgent):
    """依 state 裡的金額決定要不要跑審核——三個 template agent 都做不到這件事。"""

    reviewer: LlmAgent
    threshold: int = 10000

    async def _run_async_impl(
        self, ctx: InvocationContext
    ) -> AsyncGenerator[Event, None]:
        amount = int(ctx.session.state.get("amount", 0))
        if amount < self.threshold:
            # 小額直接放行，連模型都不用叫
            yield Event(
                author=self.name,
                content=types.Content(
                    role="model",
                    parts=[types.Part(text=f"金額 {amount} 未達門檻，自動核准。")],
                ),
            )
            return
        # 大額才交給 reviewer
        async for ev in self.reviewer.run_async(ctx):
            yield ev


from google.genai import types

conditional = ConditionalAgent(
    name="conditional",
    reviewer=LlmAgent(
        name="reviewer", model=get_model(),
        instruction="這是一筆大額支出，請給一句審核意見，繁體中文。",
    ),
)
print("✅ 自訂 agent 建立成功:", conditional.name)

✅ 自訂 agent 建立成功: conditional


In [14]:
for amount in (3000, 55000):
    r_c = InMemoryRunner(agent=conditional, app_name="day16")
    sid_c = await new_session(r_c, state={"amount": amount})
    print(f"金額 {amount:,}：", await ask(r_c, "請審核", session_id=sid_c))

金額 3,000： 金額 3000 未達門檻，自動核准。


金額 55,000： 此筆支出金額龐大，為確保公司財務穩健與合規性，請相關單位補充提供**詳細的成本效益評估、市場詢比價紀錄及高階主管核決授權證明**，暫緩撥款，待齊備後再行複審。


**能動，但你會發現這其實就是在手寫 Graph 引擎。**
這正是 ADK 2.0 推出 `Workflow` 的原因——把這件事變成宣告式的。

> **Graph 對應寫法**：`ctx.route` + dict 邊（Day 13），十行搞定，
> 而且分支結構是可讀、可測試的。

## 5. 怎麼選

| 你的狀況 | 用哪個 |
|---|---|
| 步驟固定，後面要用前面的結果 | `SequentialAgent` |
| 幾件事互不相干，想省時間 | `ParallelAgent` |
| 要反覆改進到達標 | `LoopAgent` + `exit_loop` |
| **有條件分支** | ❌ 三個都做不到 → **Graph**（Day 13） |
| **要細緻控制誰等誰** | ❌ → **`JoinNode`**（Day 13） |
| 順序讓模型自己決定 | → LLM 委派（Day 18） |
| 需要跨語言（Go / TS） | **只能用這三個** |

## 6. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `Agent 'x' already has a parent agent` | **一個 agent 只能有一個父節點**，用工廠函式產新實例 |
| pipeline 中間某站拿到空白 | 上游沒設 `output_key`，或 `{key}` 少了問號直接炸掉 |
| `LoopAgent` 跑不完 / 帳單暴增 | 沒有 agent 呼叫 `exit_loop`，只靠 `max_iterations` 擋 |
| 平行分支讀不到彼此的結果 | **設計如此**——要交流得用 Graph 的 fan-out/join |
| 想加 if/else 卻加不進去 | template agent 做不到，改用 Graph |

## 7. 動手練習

1. 把第 1 節的 `polisher` 拿掉改成兩站，比較輸出品質差多少。
2. 把 `synthesizer` 的 `{cost?}` 故意拼錯成 `{costs?}`，
   確認問號救了你（不會炸，只是拿到空字串）。
3. 把 `critic` 的 `tools=[exit_loop]` 拿掉，確認它一定跑滿 4 輪。
4. 把第 4 節的 `ConditionalAgent` 改寫成 Day 13 的 `Workflow` + `ctx.route`，
   比較兩者的程式碼行數與可讀性。

## 本日回顧

- **這三個在 2.8.0 已 deprecated**，但仍可用，而且是**跨語言共通的原語**。
- **`SequentialAgent` 靠 `output_key` + `{key?}` 傳資料**——問號一定要加。
- **⚠️ 一個 agent 只能有一個父節點**，換編排方式要用工廠函式產新實例。
- **`ParallelAgent` 的分支看不到彼此的 state**；標準骨架是
  `SequentialAgent(ParallelAgent(...), 匯總者)`。
- **⚠️ `LoopAgent` 自己不會停**——停止是 sub_agent 呼叫 `exit_loop` 的責任，
  `max_iterations` 是唯一的安全閥。
- **要條件分支就得換 Graph**；自己繼承 `BaseAgent` 等於手寫 Graph 引擎。

---
**下一天 → `../day17_multi_agent_patterns/`**